In [4]:
import torch
import torch.nn as nn
from torchsummary import summary

#搭建神经网络
class ModelDemo(nn.Module):
    #1在init魔法方法中 完成初始化
    def __init__(self):
        #1.1初始化父类成员
        super().__init__()
        #1.2搭建隐藏层+输出层
        self.linear1 = nn.Linear(in_features=3,out_features=3)
        self.linear2 = nn.Linear(in_features=3,out_features=2)
        self.output = nn.Linear(in_features=2,out_features=2)
        #1.3初始化隐藏层
        nn.init.xavier_normal_(self.linear1.weight)
        nn.init.zeros_(self.linear1.bias)

        nn.init.kaiming_normal_(self.linear2.weight)
        nn.init.zeros_(self.linear2.bias)
    #2前向传播
    def forward(self,x):
        #第一层隐藏层 加权求和+激活函数
        x = torch.sigmoid(self.linear1(x))

        x = torch.relu(self.linear2(x))

        x = torch.softmax(self.output(x),dim=-1)

        return x
    #模型训练
def train():
    #创建模型对象
    my_model = ModelDemo()     #实例化模型，底层会自动调用forward
    #创建数据集样本
    data = torch.randn(size=(5,3))
    #调用模型
    output = my_model(data)
    #计算查看模型参数

    print('=============计算模型参数===========')
    #参1神经网络模型对象 2输入数据梯度
    summary(my_model, input_size=(3,), device="cpu")
    
if __name__ == '__main__':
    train()
# ==========================================
# 执行入口
# ==========================================
if __name__ == '__main__':
    train()

# ==========================================
# 附录：模型配置与避坑备忘录 (可做学习笔记使用)
# ==========================================
"""
🧠 核心模块一：隐藏层配置准则 (激活函数与初始化绑定)
--------------------------------------------------
- 配置 1：ReLU 家族 + Kaiming 初始化 (现代网络首选)
  * 首选方案：激活函数用 ReLU，初始化匹配 Kaiming (He 初始化)。
  * 应对死区：若训练发生 Dead ReLU（梯度为0），替换为 Leaky ReLU。
  > 📝 个人批注：

- 配置 2：S型函数家族 + Xavier 初始化 (传统/备选)
  * 备选方案：必须用时，优先选零均值的 Tanh，匹配 Xavier (Glorot 初始化)。
  * 避免使用：隐藏层尽量少用 Sigmoid（极易梯度消失）。
  > 📝 个人批注：


🎯 核心模块二：输出层配置准则 (由任务驱动)
--------------------------------------------------
- 二分类任务：使用 Sigmoid 
  > 📝 个人批注：
- 多分类任务：使用 Softmax 
  > 📝 个人批注：
- 回归任务：使用 Identity (无激活函数)
  > 📝 个人批注：


🐛 核心模块三：PyTorch 常见报错避坑指南
--------------------------------------------------
1. 大小写错误 (AttributeError)
   * 现象：'ModelDemo' object has no attribute 'Linear1'
   * 排查：严查 __init__ 与 forward 中网络层命名（如 linear1）大小写是否一致。
   > 📝 个人批注：

2. 维度不匹配 (RuntimeError: mat1 and mat2 shapes cannot be multiplied)
   * 排查：第一层的 in_features 必须与输入数据的特征维度绝对相等。
   > 📝 个人批注：

3. 设备不匹配 (RuntimeError: Expected all tensors to be on the same device)
   * 现象：found at least two devices, cpu and cuda:0!
   * 排查：数据和模型必须同在 CPU 或同在 GPU。
   * 特殊坑点：torchsummary 默认在 GPU 造测试数据，若模型还在 CPU，必须调用 summary(..., device="cpu")。
   > 📝 个人批注：
"""

=============计算模型参数===========
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                    [-1, 3]              12
            Linear-2                    [-1, 2]               8
            Linear-3                    [-1, 2]               6
Total params: 26
Trainable params: 26
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
